# ContractRisk — calibración del juez final

Sesión nueva de Colab con GPU T4. Ejecutar en orden. Evalúa veinte respuestas ficticias revisadas por una persona y seis pares de verbosidad; no vuelve a ejecutar el sistema contractual ni carga su gold final.

El juez candidato es Qwen2.5-7B-Instruct con NF4, distinto del generador 3B pero de la misma familia. Verifica un kernel CUDA NF4 antes de cargar el checkpoint. Si no es compatible se conserva el fallo, sin cambiar versiones ni modelo automáticamente.

La rúbrica exige explicación: si está vacía, nota máxima 2. Es una instrucción al modelo, no un ajuste de su nota. Se conservan puntuaciones originales, incluidas las equivocadas, y como máximo un reintento solo por formato.

Habilitar acceso de este notebook al secreto `WANDB_API_KEY`. No pegar claves en celdas. Esta ejecución no modifica main ni el Space. Superar el filtro numérico requiere todavía auditoría de las razones del juez.


In [ ]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_final_judge_calibration_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError(f'Proceso terminó con código {status}; consultar {logfile}')

from google.colab import userdata
try:
    key = userdata.get('WANDB_API_KEY')
    if not key:
        raise ValueError('Empty secret')
    os.environ['WANDB_API_KEY'] = key
    del key
except Exception:
    raise RuntimeError('Habilita el acceso de ESTE notebook a WANDB_API_KEY en el panel de secretos de Colab. No pegues ni imprimas la clave.') from None
print('Secreto accesible; su valor no se imprime. La conexión se validará antes de cargar modelos.')


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r',
    'final/requirements-calibration.txt'], 'installation.log')
import torch
assert torch.cuda.is_available(), 'Selecciona GPU T4 y una sesión nueva.'
print('GPU:', torch.cuda.get_device_name(0))
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('20 anclas aprobadas; hashes y rúbrica comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
RESULTS = ROOT / 'judge_calibration_results'
run_visible([sys.executable, '-u', 'final/run_judge_calibration.py',
    '--output', str(RESULTS), '--require-wandb'], 'judge_calibration.log')


## Resultado y conservación de evidencia

Consultar metadata y resumen. `completed_gate_failed` significa que el juez no superó la calibración, aunque no haya errores de ejecución. No volver a correr para escoger notas más favorables ni sustituir puntuaciones.

`judge_validated_for_final=false` se conserva hasta auditar las razones reales. Las últimas celdas también permiten guardar una ejecución que falló.


In [ ]:
RESULTS = ROOT / 'judge_calibration_results'
for name in ['metadata.json', 'calibration_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, RESULTS / log.name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
shutil.copy2(ROOT / 'final/judge_calibration_config.json', RESULTS / 'judge_calibration_config.json')
package = shutil.make_archive(str(ROOT / 'contractrisk_final_calibracion_juez_resultados'), 'zip', RESULTS)
files.download(package)
